[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gustavoespi27/comida-chilena-cnn/blob/main/NB05_CNN_Comida_Chilena.ipynb)
# NB05 — Construcción y entrenamiento de una CNN: comida chilena

**Dataset del proyecto:** fotografías de 8 platos típicos chilenos (`cazuela`, `chacarero`, `completo`, `empanada`, `humitas`, `mote_con_huesillo`, `pastel_de_choclo`, `sopaipilla`), preparado y documentado en el NB03.

Adaptación de la guía *NB05 — Construcción y entrenamiento de una CNN con un dataset propio* (Taller Fundamentos Data Science Python, Machine Learning 2026).

## Propósito y continuidad pedagógica

En el ejercicio guiado anterior construimos y evaluamos una CNN utilizando un dataset común de imágenes de piedra, papel o tijeras. **En este notebook cada grupo aplicará ese aprendizaje a su propio problema de clasificación de imágenes**, con las categorías y los datos que haya seleccionado y preparado para su Proyecto Integrador.

El objetivo es construir una primera CNN, entrenarla, examinar sus resultados, contrastar una segunda configuración y conservar un modelo reproducible. No se espera que todos los grupos obtengan las mismas métricas: los resultados dependerán del dataset, su calidad, su tamaño y la dificultad del problema.

**Antes de comenzar:** disponer de un conjunto de imágenes organizado por clases, revisar que las etiquetas sean correctas y registrar el origen y las características de los datos. Este notebook descarga el dataset desde el repositorio de GitHub [`gustavoespi27/comida-chilena-cnn`](https://github.com/gustavoespi27/comida-chilena-cnn) y utiliza Google Drive solo para guardar los modelos entrenados.

**Criterio metodológico:** mantener separadas las imágenes de entrenamiento, validación y, cuando se disponga de ellas, prueba. Utilizar validación para comparar arquitecturas y reservar prueba para estimar el desempeño final del modelo elegido.


## 1. Preparación del entorno


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)


## 2. Conectar Google Drive (para guardar los modelos)

Los archivos creados en Colab se borran al cerrar la sesión. Por eso los modelos entrenados se guardan en `MyDrive/MachineLearning2026`. Autorice el acceso a Drive cuando Colab lo solicite.


In [ ]:
# Conectar Google Drive para GUARDAR los modelos entrenados.
# Los archivos creados en Colab se borran al cerrar la sesión; en Drive se conservan.
# Fuera de Colab, los modelos se guardan en la carpeta local "modelos".
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CARPETA_MODELOS = "/content/drive/MyDrive/MachineLearning2026"
except ImportError:
    CARPETA_MODELOS = "modelos"

os.makedirs(CARPETA_MODELOS, exist_ok=True)
print("Los modelos se guardarán en:", CARPETA_MODELOS)


## 3. Definir el dataset propio del proyecto

Utilice las imágenes que su grupo seleccionó y organizó para su problema de clasificación. La carpeta debe contener **una subcarpeta por clase**; los nombres de las carpetas determinarán las etiquetas del modelo.

Estructura del dataset de comida chilena:

```text
dataset_comida_chilena/
├── cazuela/
├── chacarero/
├── completo/
├── empanada/
├── humitas/
├── mote_con_huesillo/
├── pastel_de_choclo/
└── sopaipilla/
```

Compruebe que cada clase tenga imágenes válidas y una cantidad suficiente para realizar entrenamiento y validación. La siguiente celda obtiene el dataset: si el notebook se ejecuta dentro del repositorio usa esa carpeta; en Colab descarga el repositorio con `git clone` (solo la primera vez).

**Atención:** la división automática que se utiliza más adelante crea entrenamiento y validación a partir de las imágenes presentes en `DATASET_DIR`. Si su proyecto dispone de un conjunto de prueba independiente, manténgalo fuera de esa carpeta para evitar filtraciones de datos.


In [ ]:
# Obtener el dataset desde GitHub
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/gustavoespi27/comida-chilena-cnn.git"

if Path("dataset_comida_chilena").is_dir():
    BASE_DIR = Path(".")                     # el notebook ya está dentro del repositorio
else:
    BASE_DIR = Path("comida-chilena-cnn")
    if not BASE_DIR.is_dir():                # clonar solo la primera vez
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(BASE_DIR)], check=True)

DATASET_DIR = BASE_DIR / "dataset_comida_chilena"
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"No se encontró el dataset: {DATASET_DIR}")
print("Dataset:", DATASET_DIR.resolve())
DATASET_DIR = str(DATASET_DIR)  # image_dataset_from_directory recibe la ruta como texto

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


## 4. Crear conjuntos de entrenamiento y validación


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("Clases:", class_names)
print("Número de clases:", NUM_CLASSES)


## 5. Visualizar imágenes del dataset


In [ ]:
plt.figure(figsize=(9, 9))

for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.tight_layout()
plt.show()


## 6. Optimizar el pipeline de datos


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


## 7. Data Augmentation

Las transformaciones se aplicarán solamente durante el entrenamiento.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10)
], name="data_augmentation")


## 8. Construcción de la CNN

La arquitectura combinará:

**Convolución → activación → pooling**

para extraer progresivamente características de las imágenes.

Posteriormente, las características serán utilizadas por capas densas para realizar la clasificación.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.30),

    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()


## 9. Analizar la arquitectura

Observe especialmente:

- dimensiones de entrada y salida;
- cantidad de filtros;
- reducción espacial después de cada pooling;
- cantidad de parámetros;
- capa de clasificación final.


In [ ]:
print("Parámetros totales:", model.count_params())


## 10. Compilar la CNN


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


## 11. Entrenamiento inicial


In [ ]:
EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


## 12. Curvas de accuracy


In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="Entrenamiento")
plt.plot(history_df["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Accuracy durante el entrenamiento")
plt.legend()
plt.show()


## 13. Curvas de pérdida


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Entrenamiento")
plt.plot(history_df["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Pérdida durante el entrenamiento")
plt.legend()
plt.show()


## 14. Evaluación sobre validación


In [ ]:
val_loss, val_accuracy = model.evaluate(
    val_ds,
    verbose=0
)

print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")


## 15. Realizar predicciones


In [ ]:
for images, labels in val_ds.take(1):
    predictions = model.predict(images, verbose=0)

    predicted_labels = np.argmax(predictions, axis=1)
    confidence = np.max(predictions, axis=1)

    resultados = pd.DataFrame({
        "Clase_real": [class_names[int(x)] for x in labels.numpy()],
        "Clase_predicha": [class_names[int(x)] for x in predicted_labels],
        "Confianza": confidence
    })

resultados.head(10)


## 16. Visualizar predicciones


In [ ]:
for images, labels in val_ds.take(1):
    predictions = model.predict(images, verbose=0)
    predicted_labels = np.argmax(predictions, axis=1)
    confidence = np.max(predictions, axis=1)

    plt.figure(figsize=(10, 10))

    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)

        plt.imshow(images[i].numpy().astype("uint8"))

        real = class_names[int(labels[i])]
        pred = class_names[int(predicted_labels[i])]
        conf = confidence[i]

        plt.title(
            f"Real: {real}\nPred: {pred} ({conf:.1%})"
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()


## 17. Matriz de confusión


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images, verbose=0)
    predicted = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted)

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("Matriz de confusión")
plt.show()


## 18. Métricas por clase


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)


## 19. Guardar el modelo

El modelo guardado podrá utilizarse posteriormente para continuar el proyecto.


In [ ]:
MODEL_PATH = f"{CARPETA_MODELOS}/cnn_comida_chilena_inicial_v1.keras"

model.save(MODEL_PATH)

print("Modelo guardado en:")
print(MODEL_PATH)


## 20. Experimentación con la arquitectura

Construya una segunda CNN modificando **uno o dos elementos** de la arquitectura inicial.

Puede experimentar con:

- número de filtros;
- cantidad de bloques convolucionales;
- tamaño del kernel;
- Dropout;
- cantidad de neuronas densas;
- learning rate;
- número de épocas.

Evite modificar simultáneamente demasiados elementos, ya que dificultará interpretar qué cambio produjo el resultado.


In [ ]:
modelo_experimental = keras.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(256, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.40),

    layers.Dense(NUM_CLASSES, activation="softmax")
])

modelo_experimental.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

modelo_experimental.summary()


## 21. Entrenar el modelo experimental


In [ ]:
history_exp = modelo_experimental.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

exp_loss, exp_accuracy = modelo_experimental.evaluate(
    val_ds,
    verbose=0
)

print(f"Validation accuracy modelo experimental: {exp_accuracy:.4f}")


## 22. Comparar modelos


In [ ]:
comparacion = pd.DataFrame({
    "Modelo": [
        "CNN inicial",
        "CNN experimental"
    ],
    "Validation_accuracy": [
        val_accuracy,
        exp_accuracy
    ],
    "Parametros": [
        model.count_params(),
        modelo_experimental.count_params()
    ]
})

comparacion


## 23. Actividad

A partir del dataset de comida chilena:

1. Construya una CNN funcional.
2. Identifique las capas convolucionales y de pooling.
3. Registre la cantidad total de parámetros.
4. Entrene el modelo y analice las curvas de accuracy y loss.
5. Identifique posibles señales de overfitting o underfitting.
6. Evalúe el comportamiento por clase.
7. Construya una segunda arquitectura.
8. Compare ambas CNN utilizando evidencia.
9. Determine cuál utilizaría como punto de partida para la siguiente etapa del proyecto.
10. Guarde el modelo y registre claramente su versión.


### 23.1 Capas convolucionales, pooling y parámetros (puntos 1 a 3)

In [ ]:
def describir_capas(modelo):
    filas = []
    for capa in modelo.layers:
        tipo = type(capa).__name__
        if tipo in ("Conv2D", "MaxPooling2D", "GlobalAveragePooling2D", "Dense", "Dropout"):
            filas.append({
                "capa": capa.name,
                "tipo": tipo,
                "salida": str(capa.output.shape),
                "filtros/neuronas": getattr(capa, "filters", getattr(capa, "units", "")),
                "parámetros": capa.count_params(),
            })
    return pd.DataFrame(filas)

print("CNN inicial — parámetros totales:", model.count_params())
display(describir_capas(model))
print("CNN experimental — parámetros totales:", modelo_experimental.count_params())
display(describir_capas(modelo_experimental))


### 23.2 Señales de overfitting o underfitting (puntos 4 y 5)

Se comparan las curvas de ambos modelos y se resume la última época y la mejor época de validación.

In [ ]:
def diagnostico(historial, nombre):
    h = pd.DataFrame(historial.history)
    ultima = h.iloc[-1]
    mejor_epoca = int(h["val_loss"].idxmin()) + 1
    brecha = ultima["accuracy"] - ultima["val_accuracy"]
    if ultima["accuracy"] < 0.6 and ultima["val_accuracy"] < 0.6:
        lectura = "posible underfitting (accuracy baja en ambos conjuntos)"
    elif brecha > 0.10 or mejor_epoca < len(h) - 3:
        lectura = "posible overfitting (train supera a validación o val_loss deja de bajar)"
    else:
        lectura = "sin señales fuertes de overfitting ni underfitting"
    return {"modelo": nombre, "acc_train_final": round(ultima["accuracy"], 4),
            "acc_val_final": round(ultima["val_accuracy"], 4), "brecha_train_val": round(brecha, 4),
            "mejor_epoca_val_loss": mejor_epoca, "epocas": len(h), "lectura": lectura}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for historial, nombre, estilo in [(history, "inicial", "-"), (history_exp, "experimental", "--")]:
    h = pd.DataFrame(historial.history)
    axes[0].plot(h["accuracy"], estilo, label=f"{nombre} train")
    axes[0].plot(h["val_accuracy"], estilo, label=f"{nombre} validación")
    axes[1].plot(h["loss"], estilo, label=f"{nombre} train")
    axes[1].plot(h["val_loss"], estilo, label=f"{nombre} validación")
axes[0].set_title("Accuracy"); axes[1].set_title("Loss")
for ax in axes:
    ax.set_xlabel("Época"); ax.legend()
plt.tight_layout()
plt.show()

pd.DataFrame([diagnostico(history, "CNN inicial"), diagnostico(history_exp, "CNN experimental")])


### 23.3 Comportamiento por clase de ambos modelos (punto 6)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def predecir(modelo):
    reales, predichas = [], []
    for images, labels in val_ds:
        reales.extend(labels.numpy())
        predichas.extend(np.argmax(modelo.predict(images, verbose=0), axis=1))
    return np.array(reales), np.array(predichas)

y_true_ini, y_pred_ini = predecir(model)
y_true_exp, y_pred_exp = predecir(modelo_experimental)

def f1_por_clase(y_t, y_p):
    _, _, f1, soporte = precision_recall_fscore_support(y_t, y_p, labels=range(NUM_CLASSES), zero_division=0)
    return f1, soporte

f1_ini, soporte = f1_por_clase(y_true_ini, y_pred_ini)
f1_exp, _ = f1_por_clase(y_true_exp, y_pred_exp)
por_clase = pd.DataFrame({"clase": class_names, "imagenes_validacion": soporte,
                          "f1_inicial": f1_ini.round(3), "f1_experimental": f1_exp.round(3)})
display(por_clase.sort_values("f1_inicial"))

# Pares de clases que más se confunden (modelo inicial)
cm_ini = confusion_matrix(y_true_ini, y_pred_ini, labels=range(NUM_CLASSES))
confusiones = [(class_names[i], class_names[j], int(cm_ini[i, j]))
               for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and cm_ini[i, j] > 0]
pd.DataFrame(sorted(confusiones, key=lambda x: -x[2])[:6], columns=["clase_real", "predicha_como", "cantidad"])


### 23.4 Comparación y elección del modelo (puntos 7 a 10)

Se elige el modelo con mayor *accuracy* de validación; en caso de empate se prefiere el que tiene menos parámetros. Se guarda con un nombre versionado y una ficha JSON con la información necesaria para reproducirlo.

In [ ]:
import json
from datetime import date

comparacion["F1_macro"] = [f1_ini.mean().round(4), f1_exp.mean().round(4)]
display(comparacion)

candidatos = [(val_accuracy, -model.count_params(), "CNN inicial", model),
              (exp_accuracy, -modelo_experimental.count_params(), "CNN experimental", modelo_experimental)]
acc_elegido, _, nombre_elegido, modelo_elegido = max(candidatos, key=lambda x: (round(x[0], 4), x[1]))
print(f"Modelo elegido: {nombre_elegido} (validation accuracy = {acc_elegido:.4f})")

VERSION = "v1"
RUTA_ELEGIDO = f"{CARPETA_MODELOS}/cnn_comida_chilena_elegido_{VERSION}.keras"
modelo_elegido.save(RUTA_ELEGIDO)

ficha_modelo = {
    "version": VERSION,
    "fecha": str(date.today()),
    "modelo": nombre_elegido,
    "archivo": RUTA_ELEGIDO,
    "clases_en_orden": class_names,
    "tamano_entrada": list(IMG_SIZE),
    "parametros": int(modelo_elegido.count_params()),
    "epocas": EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizador": "Adam (learning_rate=0.001)",
    "division": "80 % train / 20 % validation, seed 42",
    "validation_accuracy": round(float(acc_elegido), 4),
}
with open(f"{CARPETA_MODELOS}/cnn_comida_chilena_elegido_{VERSION}.json", "w", encoding="utf-8") as f:
    json.dump(ficha_modelo, f, ensure_ascii=False, indent=2)
ficha_modelo


### 23.5 Respuestas

> **Cómo completar estas respuestas:** los puntos 1 a 3 no cambian entre ejecuciones. Los puntos 4 a 10 dependen del entrenamiento, que varía un poco cada vez que se ejecuta; reemplace cada **[ … ]** con los valores que entregan las celdas 23.1 a 23.4 **de su propia ejecución**.

**1. CNN funcional.** Se construyó una CNN secuencial que recibe imágenes RGB de 224 × 224 y entrega 8 probabilidades (una por plato) con `softmax`. Incluye *Data Augmentation* (flip horizontal, rotación 0,10 y zoom 0,10, activos solo durante el entrenamiento) y normalización `Rescaling(1./255)`. Se entrena con Adam (lr = 0,001) y `sparse_categorical_crossentropy`, durante 20 épocas, con 860 imágenes de entrenamiento y 214 de validación (división 80/20, semilla 42).

**2. Capas convolucionales y de pooling.**

| Bloque | CNN inicial | CNN experimental |
|---|---|---|
| 1 | Conv2D 32 filtros 3×3 → MaxPooling 2×2 (224 → 112) | igual |
| 2 | Conv2D 64 filtros 3×3 → MaxPooling 2×2 (112 → 56) | igual |
| 3 | Conv2D 128 filtros 3×3 → MaxPooling 2×2 (56 → 28) | igual |
| 4 | — | Conv2D 256 filtros 3×3 → MaxPooling 2×2 (28 → 14) |
| Cierre | GlobalAveragePooling → Dense 128 → Dropout 0,30 → Dense 8 | GlobalAveragePooling → Dense 128 → Dropout **0,40** → Dense 8 |

Las convoluciones (con `padding="same"`) extraen patrones visuales y mantienen el tamaño espacial. Cada *pooling* reduce el alto y el ancho a la mitad, mientras aumenta la cantidad de filtros. `GlobalAveragePooling2D` resume cada mapa de características en un solo valor, antes de las capas densas.

**3. Cantidad de parámetros.** CNN inicial: **110.792** parámetros. CNN experimental: **422.344** (3,8 veces más). La mayor parte de la diferencia viene del cuarto bloque convolucional, que tiene 295.168 parámetros.

**4. Curvas de accuracy y loss.** CNN inicial: *accuracy* de entrenamiento final **[ … ]** y de validación **[ … ]**. La *val_loss* alcanza su mínimo en la época **[ … ]**. *[Describir: ¿las curvas suben o bajan de forma sostenida? ¿la curva de validación oscila mucho? Con solo 214 imágenes de validación, es normal que oscile.]*

**5. Overfitting o underfitting.** La celda 23.2 indica: **[ … lectura del diagnóstico … ]**. *[Guía: si la accuracy de train supera a la de validación por más de unos 10 puntos, o si val_loss deja de bajar mientras la loss de train sigue bajando, hay overfitting. Si ambas accuracy quedan bajas (por ejemplo, bajo 60 %), hay underfitting: el modelo es muy simple o necesita más épocas. El Data Augmentation y el Dropout ayudan a reducir el overfitting.]*

**6. Comportamiento por clase.** Clases con mejor F1: **[ … ]**. Clases con peor F1: **[ … ]**. Confusiones más frecuentes (celda 23.3): **[ … ]**. *[Guía: compare con lo anticipado en el NB03. Se esperan confusiones entre humitas y pastel de choclo (las dos son de choclo amarillo), entre chacarero y completo (los dos son sándwiches) y entre empanada y sopaipilla (las dos son masas doradas). Mote con huesillo es la clase con menos imágenes, aunque su aspecto (un vaso con líquido y mote) es muy distinto al resto.]*

**7. Segunda arquitectura.** La CNN experimental cambia **dos** elementos respecto de la inicial: **(a)** agrega un cuarto bloque convolucional de 256 filtros, para aprender patrones más complejos, y **(b)** aumenta el Dropout de 0,30 a 0,40, para compensar el mayor riesgo de overfitting que trae un modelo con más parámetros. Se mantuvieron iguales los datos, el optimizador, el learning rate y las épocas, para poder atribuir la diferencia a estos cambios.

**8. Comparación con evidencia.** Validation accuracy: inicial **[ … ]** frente a experimental **[ … ]**. F1 macro: **[ … ]** frente a **[ … ]**. *[Indique cuál generaliza mejor, si la diferencia es grande o pequeña (con 214 imágenes, 1 imagen equivale a 0,5 puntos de accuracy) y si el modelo experimental justifica tener 3,8 veces más parámetros.]*

**9. Modelo elegido.** Se elige **[CNN inicial / CNN experimental]** como punto de partida, porque **[ … ]**. *[El criterio automático de la celda 23.4 es la mayor validation accuracy; si hay empate, se prefiere el modelo con menos parámetros. Puede argumentar otra elección con evidencia, por ejemplo un mejor F1 en las clases difíciles.]* **Limitaciones:** el dataset es pequeño (1.074 imágenes), viene de búsquedas web (muchas fotos de recetas, con buena iluminación y emplatado, distintas a una foto casera) y no hay un conjunto de prueba independiente en este notebook. Para la siguiente etapa convendría usar *Transfer Learning* (NB06) y evaluar con un conjunto *test* reservado.

**10. Modelo guardado y versión.** El modelo inicial se guardó como `cnn_comida_chilena_inicial_v1.keras` (sección 19). El modelo elegido se guardó como `cnn_comida_chilena_elegido_v1.keras`, junto con su ficha `cnn_comida_chilena_elegido_v1.json`, que registra la versión, la fecha, las clases en orden, el tamaño de entrada, los parámetros, las épocas, el optimizador, la división de datos y la validation accuracy. Todo quedó en `MyDrive/MachineLearning2026/`. Si se entrena una nueva versión, se debe cambiar `VERSION = "v2"` para no sobrescribir la anterior.



## 24. Mejora adicional: Transfer Learning con MobileNetV2

Las CNN entrenadas desde cero quedan limitadas por el tamaño del dataset: con 860 imágenes de entrenamiento no alcanzan a aprender bien qué distingue a cada plato. Se probaron de forma automática distintas combinaciones de hiperparámetros (bloques, filtros, BatchNorm, dropout, learning rate, batch size, *weight decay* y Data Augmentation) y la mejor CNN desde cero llegó a cerca de **59 %** de *validation accuracy* con *val_loss* ≈ **1,16** (la CNN experimental entrenada hasta 60 épocas con *early stopping*).

Para acercarse al 100 % se usa **Transfer Learning con MobileNetV2**, la misma técnica y red de la guía NB06: se parte de una red ya entrenada con 1,2 millones de imágenes de ImageNet, que ya sabe reconocer bordes, texturas y formas. Primero se entrena solo la capa final para los 8 platos y luego se ajustan las últimas capas con un learning rate muy bajo (*fine-tuning*).

Respecto de la receta del NB06 (10 + 10 épocas, *fine-tuning* desde la capa 100) se ajustaron tres hiperparámetros, todos con herramientas de Keras: más épocas con `EarlyStopping` (se detiene cuando *val_loss* deja de bajar y conserva los mejores pesos), `ReduceLROnPlateau` (baja el learning rate si el modelo se estanca) y *fine-tuning* solo de las últimas 30 capas, sin modificar las de BatchNormalization.

| Modelo | val_accuracy | val_loss |
|---|---|---|
| CNN inicial (NB05, 20 épocas) | ≈ 0,52 | ≈ 1,32 |
| CNN experimental, 60 épocas + *early stopping* | ≈ 0,59 | ≈ 1,16 |
| MobileNetV2, receta del NB06 (10 + 10 épocas) | ≈ 0,967 | ≈ 0,095 |
| **MobileNetV2 con hiperparámetros ajustados** | **≈ 0,981** | **≈ 0,063** |

*Valores de las pruebas previas (misma división 80/20, semilla 42). Su ejecución puede variar levemente.*

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False   # se congelan los pesos aprendidos en ImageNet

inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
modelo_tl = keras.Model(inputs, outputs, name="mobilenetv2_comida_chilena")

# Hiperparámetros ajustados
EPOCHS_TRANSFER = 25
EPOCHS_FINE = 15
callbacks_tl = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

modelo_tl.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_transfer = modelo_tl.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TRANSFER, callbacks=callbacks_tl)


In [ ]:
# Fine-tuning: se descongelan las últimas 30 capas, sin tocar BatchNormalization
base_model.trainable = True
FINE_TUNE_AT = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
print("Capas totales:", len(base_model.layers))
print("Capas entrenables:", sum(layer.trainable for layer in base_model.layers))

modelo_tl.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5),   # learning rate muy bajo
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_fine = modelo_tl.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, callbacks=callbacks_tl)

tl_loss, tl_accuracy = modelo_tl.evaluate(val_ds, verbose=0)
print(f"MobileNetV2 + fine-tuning -> validation accuracy: {tl_accuracy:.4f} | validation loss: {tl_loss:.4f}")


In [ ]:
# Curvas de ambas etapas y comparación con las CNN desde cero
hist = {k: history_transfer.history[k] + history_fine.history[k] for k in ("accuracy", "val_accuracy", "loss", "val_loss")}
corte = len(history_transfer.history["loss"])

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for eje, metrica in zip(ejes, ("accuracy", "loss")):
    eje.plot(hist[metrica], label=f"train {metrica}")
    eje.plot(hist[f"val_{metrica}"], label=f"val {metrica}")
    eje.axvline(corte - 0.5, color="gray", linestyle="--", label="inicio fine-tuning")
    eje.set_title(f"MobileNetV2 - {metrica}"); eje.set_xlabel("Época"); eje.legend()
plt.tight_layout(); plt.show()

y_true_tl, y_pred_tl = predecir(modelo_tl)
print(classification_report(y_true_tl, y_pred_tl, target_names=class_names, digits=3))

resumen_final = pd.DataFrame({
    "modelo": ["CNN inicial", "CNN experimental", "MobileNetV2 + fine-tuning"],
    "val_accuracy": [round(val_accuracy, 4), round(exp_accuracy, 4), round(tl_accuracy, 4)],
    "val_loss": [round(model.evaluate(val_ds, verbose=0)[0], 4), round(modelo_experimental.evaluate(val_ds, verbose=0)[0], 4), round(tl_loss, 4)],
    "parametros": [model.count_params(), modelo_experimental.count_params(), modelo_tl.count_params()],
})
display(resumen_final)


In [ ]:
# Guardar el modelo de Transfer Learning con su ficha
RUTA_TL = f"{CARPETA_MODELOS}/mobilenetv2_comida_chilena_{VERSION}.keras"
modelo_tl.save(RUTA_TL)
ficha_tl = {
    "version": VERSION,
    "fecha": str(date.today()),
    "modelo": "MobileNetV2 (ImageNet) + fine-tuning de las últimas 30 capas",
    "archivo": RUTA_TL,
    "clases_en_orden": class_names,
    "tamano_entrada": list(IMG_SIZE),
    "preprocesamiento": "MobileNetV2 preprocess_input",
    "parametros": int(modelo_tl.count_params()),
    "entrenamiento": f"{len(history_transfer.history['loss'])} épocas (lr 0.001) + {len(history_fine.history['loss'])} de fine-tuning (lr 1e-5), early stopping",
    "division": "80 % train / 20 % validation, seed 42",
    "validation_accuracy": round(float(tl_accuracy), 4),
    "validation_loss": round(float(tl_loss), 4),
}
with open(f"{CARPETA_MODELOS}/mobilenetv2_comida_chilena_{VERSION}.json", "w", encoding="utf-8") as f:
    json.dump(ficha_tl, f, ensure_ascii=False, indent=2)
ficha_tl


**Conclusión.** El Transfer Learning supera ampliamente a las CNN entrenadas desde cero: reutiliza lo que MobileNetV2 aprendió de millones de imágenes y solo adapta la parte final a los 8 platos chilenos. Ajustar las épocas, el learning rate y las capas del *fine-tuning* mejoró la receta base del NB06 de ≈ 96,7 % a ≈ 98,1 % de *accuracy* y bajó la *loss* de ≈ 0,095 a ≈ 0,063. Las CNN de las secciones anteriores siguen siendo el resultado pedido por la guía NB05; este modelo es una mejora que adelanta el NB06. Con 214 imágenes de validación, cada error equivale a 0,5 puntos de *accuracy*, por lo que conviene confirmar el resultado con un conjunto *test* independiente (como el del NB03).

## 24. Cierre y continuidad del Proyecto Integrador

Al finalizar este notebook, cada grupo debe contar con **una CNN inicial entrenada sobre su propio dataset**, evidencia de su comportamiento y una comparación razonada con una segunda configuración.

**Dataset propio preparado → arquitectura CNN → entrenamiento → validación → métricas por clase → comparación de modelos → versión inicial seleccionada**

Conserve el modelo elegido, los parámetros de entrenamiento, la definición de clases y las métricas obtenidas. Estos elementos permitirán continuar con la mejora, la integración y el eventual despliegue de una aplicación de clasificación de imágenes.

**Reflexión final:** ¿qué evidencia respalda la elección del modelo y qué limitaciones del dataset o del entrenamiento deben abordarse antes de utilizarlo con imágenes nuevas?
